# K-Means Clustering Algorithm

## Learning Objectives
- Understand clustering as an **unsupervised learning** problem — no labels $y^{(i)}$ are given
- Implement the k-means algorithm: initialise centroids, then repeat assign → update
- Define the **distortion function** $J(c, \mu) = \sum_{i=1}^m \|x^{(i)} - \mu_{c^{(i)}}\|^2$
- Understand that k-means is **coordinate descent on $J$** and must converge
- Understand susceptibility to local optima and the multi-restart remedy

## Problem Statement

Given unlabelled training set $\{x^{(1)}, \ldots, x^{(m)}\}$ with $x^{(i)} \in \mathbb{R}^n$, group the data into $k$ cohesive clusters.

**K-means algorithm:**

1. Initialise cluster centroids $\mu_1, \mu_2, \ldots, \mu_k \in \mathbb{R}^n$ randomly
2. Repeat until convergence:
   - **Assign** each example to its nearest centroid: $c^{(i)} := \arg\min_j \|x^{(i)} - \mu_j\|^2$
   - **Update** each centroid to the mean of its assigned points: $\mu_j := \dfrac{\sum_{i=1}^m \mathbf{1}\{c^{(i)}=j\}\, x^{(i)}}{\sum_{i=1}^m \mathbf{1}\{c^{(i)}=j\}}$

**Distortion function:**
$$J(c, \mu) = \sum_{i=1}^m \|x^{(i)} - \mu_{c^{(i)}}\|^2$$

K-means is coordinate descent on $J$: the assign step minimises $J$ over $c$ (holding $\mu$ fixed), and the update step minimises $J$ over $\mu$ (holding $c$ fixed). Therefore $J$ decreases monotonically and the algorithm converges.

## 1. K-Means Implementation

In [ ]:
import numpy as np

def kmeans(X, k, max_iters=100, seed=None):
    rng = np.random.default_rng(seed)
    m   = X.shape[0]

    # Initialise: pick k random data points as centroids
    mu = X[rng.choice(m, k, replace=False)].copy().astype(float)

    history = []   # (centroids, assignments, distortion) per iteration

    for _ in range(max_iters):
        # Assign step: c^(i) = argmin_j ||x^(i) - mu_j||^2
        dists = np.linalg.norm(X[:, None, :] - mu[None, :, :], axis=2)  # (m, k)
        c     = np.argmin(dists, axis=1)                                  # (m,)

        J = np.sum(np.min(dists, axis=1) ** 2)
        history.append((mu.copy(), c.copy(), J))

        # Update step: mu_j = mean of points assigned to cluster j
        mu_new = np.array([
            X[c == j].mean(axis=0) if (c == j).any() else mu[j]
            for j in range(k)
        ])

        if np.allclose(mu, mu_new):
            break
        mu = mu_new

    return mu, c, history


# Smoke test
np.random.seed(0)
X_test = np.vstack([
    np.random.randn(20, 2) + [0, 0],
    np.random.randn(20, 2) + [5, 5],
    np.random.randn(20, 2) + [0, 5],
])
mu_final, c_final, hist = kmeans(X_test, k=3, seed=1)
print(f'Converged in {len(hist)} iterations')
print(f'Final distortion J = {hist[-1][2]:.2f}')
print(f'Centroids:\n{mu_final}')

## 2. Step-by-Step Iteration Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
X = np.vstack([
    np.random.randn(25, 2) * 0.8 + [0, 0],
    np.random.randn(25, 2) * 0.8 + [5, 1],
    np.random.randn(25, 2) * 0.8 + [2, 5],
])

mu_final, c_final, hist = kmeans(X, k=3, seed=7)

colors  = ['#2166ac', '#d6604d', '#4dac26']
n_show  = min(6, len(hist))
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for idx, ax in enumerate(axes[:n_show]):
    mu_snap, c_snap, J_snap = hist[idx]

    for j in range(3):
        pts = X[c_snap == j]
        if len(pts):
            ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=40, alpha=0.7)
        ax.scatter(mu_snap[j, 0], mu_snap[j, 1],
                   marker='X', s=250, c=colors[j],
                   edgecolors='k', linewidths=1.5, zorder=5)

    ax.set_title(f'Iteration {idx+1}   $J = {J_snap:.1f}$', fontsize=10)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('K-Means Iterations (✕ = centroid)\nDistortion $J$ decreases each step',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Distortion $J$ as Coordinate Descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
X = np.vstack([
    np.random.randn(30, 2) + [0, 0],
    np.random.randn(30, 2) + [6, 0],
    np.random.randn(30, 2) + [3, 5],
])

_, _, hist = kmeans(X, k=3, seed=5)
J_vals = [h[2] for h in hist]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: J vs iteration
ax = axes[0]
ax.plot(range(1, len(J_vals)+1), J_vals, 'b-o', lw=2.5, ms=7)
ax.set_xlabel('Iteration'); ax.set_ylabel('Distortion $J(c, \\mu)$')
ax.set_title('Distortion $J$ decreases monotonically\n'
             'K-means = coordinate descent on $J$')
ax.text(0.55, 0.95,
        'Assign step: minimise $J$ over $c$\n(hold $\\mu$ fixed)\n\n'
        'Update step: minimise $J$ over $\\mu$\n(hold $c$ fixed)',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# Right: final clustering
ax = axes[1]
colors = ['#2166ac', '#d6604d', '#4dac26']
mu_f, c_f, _ = hist[-1]
for j in range(3):
    pts = X[c_f == j]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=50, alpha=0.75, label=f'Cluster {j+1}')
    ax.scatter(mu_f[j, 0], mu_f[j, 1], marker='X', s=280,
               c=colors[j], edgecolors='k', linewidths=1.5, zorder=5)
ax.set_title(f'Final clustering after {len(hist)} iterations\n$J_{{\\text{{final}}}} = {J_vals[-1]:.1f}$')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=9)

fig.suptitle('K-Means Convergence via Distortion Function', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Local Optima and Multi-Restart Strategy

$J$ is non-convex, so k-means can converge to different local minima depending on initialisation.

**Remedy:** run k-means many times with different random initialisations; keep the result with the lowest $J$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
X = np.vstack([
    np.random.randn(20, 2) * 0.7 + [0, 0],
    np.random.randn(20, 2) * 0.7 + [4, 0],
    np.random.randn(20, 2) * 0.7 + [2, 3.5],
    np.random.randn(20, 2) * 0.7 + [6, 3.5],
])

n_restarts = 20
results    = [kmeans(X, k=4, seed=s) for s in range(n_restarts)]
J_finals   = [r[2][-1][2] for r in results]

best_idx  = int(np.argmin(J_finals))
worst_idx = int(np.argmax(J_finals))

colors = ['#2166ac', '#d6604d', '#4dac26', '#7b2d8b']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: J distribution across restarts
ax = axes[0]
ax.hist(J_finals, bins=10, color='steelblue', edgecolor='k', alpha=0.8)
ax.axvline(J_finals[best_idx],  color='green', ls='--', lw=2, label=f'Best  J={J_finals[best_idx]:.1f}')
ax.axvline(J_finals[worst_idx], color='red',   ls='--', lw=2, label=f'Worst J={J_finals[worst_idx]:.1f}')
ax.set_xlabel('Final distortion $J$'); ax.set_ylabel('Count')
ax.set_title(f'Distribution of $J$ over {n_restarts} random restarts\n(k=4)')
ax.legend(fontsize=9)

for ax, idx, label in [
    (axes[1], best_idx,  f'Best initialisation (seed={best_idx})\n$J={J_finals[best_idx]:.1f}$'),
    (axes[2], worst_idx, f'Worst initialisation (seed={worst_idx})\n$J={J_finals[worst_idx]:.1f}$'),
]:
    mu_f, c_f, hist_f = results[idx]
    for j in range(4):
        pts = X[c_f == j]
        ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=45, alpha=0.75)
        ax.scatter(mu_f[j, 0], mu_f[j, 1], marker='X', s=260,
                   c=colors[j], edgecolors='k', linewidths=1.5, zorder=5)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

fig.suptitle('Local Optima in K-Means — Multi-Restart Strategy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="374"
     viewBox="0 0 780 374" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Initialise -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Initialise</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">centroids &#x3bc;&#x2081;,...,&#x3bc;&#x2096;</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >pick k random data points; multiple restarts avoid local optima</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >assign step: c&#x2071; = argmin&#x2c7; ||x&#x2071;&#x2212;&#x3bc;&#x2c7;||&#xb2;</text>

  <!-- Row 2: Assign -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="140" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Assign step</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >minimises J over c (holding &#x3bc; fixed) &#x2014; reduces distortion</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >update step: &#x3bc;&#x2c7; = mean of {x&#x2071;: c&#x2071;=j}</text>

  <!-- Row 3: Update -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Update step</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >minimises J over &#x3bc; (holding c fixed) &#x2014; J decreases monotonically</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="282" font-size="11.5" font-style="italic" fill="#555"
        >repeat until centroids stabilise</text>

  <!-- Row 4: Convergence -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Converged</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="340" font-size="13" text-anchor="middle" fill="#333"
        >local minimum of J &#x2014; pick best restart; then EM generalises this (next notebooks)</text>
</svg>
""")

## Summary

| Concept | Formula / Description | Key Insight |
|---|---|---|
| Distortion function | $J(c,\mu) = \sum_i \|x^{(i)} - \mu_{c^{(i)}}\|^2$ | The objective k-means minimises |
| Assign step | $c^{(i)} := \arg\min_j \|x^{(i)} - \mu_j\|^2$ | Minimises $J$ over $c$ holding $\mu$ fixed |
| Update step | $\mu_j := \frac{\sum_i \mathbf{1}\{c^{(i)}=j\} x^{(i)}}{\sum_i \mathbf{1}\{c^{(i)}=j\}}$ | Minimises $J$ over $\mu$ holding $c$ fixed |
| Convergence | $J$ decreases monotonically; terminates in finite steps | Guaranteed because assignments are finite and $J \geq 0$ |
| Local optima | Non-convex $J$; different inits give different results | Run multiple restarts; keep lowest $J$ |
| Unsupervised | No labels $y^{(i)}$ — discover structure in $x$ alone | Contrast with supervised learning |

**Key insight:** k-means is coordinate descent on the distortion $J$; it converges but to a local minimum — multiple restarts with different random initialisations are the standard remedy.